In [2]:
# 导入所需库
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
import datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 只显示error
os.environ['PROJ_LIB'] = r'D:\ProgramData\Anaconda3\envs\tensorflow210\Library\share\proj'
import csv

## 读取与预处理数据

In [3]:
import pandas as pd
import numpy as np
import rasterio
import glob
import os
from rasterio.warp import transform

# ================= 配置区 =================
# 1. 简化的训练点位表 (必须包含 Index, Alliance, X, Y)
# X, Y 必须是 WGS84 (经纬度) 坐标
simple_train_csv = r'F:/TensorFlow/xinjiang/traindata20250626_3.csv'

# 2. 遥感影像文件夹 (包含那 ~250 个单波段 tif)
tif_folder = r'F:\TensorFlow\Xinjiang_2'

# 3. 类别映射表 (保持不变)
mapping_csv = r'F:/TensorFlow/xinjiang/traindata20250626_2_class_mapping.csv'
# ==========================================

print("正在读取训练点位数据...")
points_df = pd.read_csv(simple_train_csv)

# 确保列名正确，如果不包含 Index 可以自动生成
if 'Index' not in points_df.columns:
    points_df['Index'] = range(len(points_df))

# 获取 TIF 文件列表 (必须排序，这是特征对齐的灵魂)
tif_files = sorted(glob.glob(os.path.join(tif_folder, '*.tif')))
print(f"检测到 {len(tif_files)} 个特征影像文件。")

# ==== 步骤 1: 坐标转换 (WGS84 -> 影像投影) ====
# 读取第一幅影像作为参考坐标系
with rasterio.open(tif_files[0]) as src:
    dst_crs = src.crs  # 影像的坐标系
    
print("正在进行坐标转换 (WGS84 -> 影像投影坐标)...")
# 假设 X是经度, Y是纬度。WGS84 的 EPSG 代码是 4326
# rasterio.warp.transform 输入是 (src_crs, dst_crs, xs, ys)
# 注意：transform 返回的是 tuple (new_xs, new_ys)
tgt_x, tgt_y = transform({'init': 'epsg:4326'}, dst_crs, points_df['X'].values, points_df['Y'].values)

# 将转换后的坐标生成 sample 列表: [(x1, y1), (x2, y2), ...]
sample_coords = list(zip(tgt_x, tgt_y))

# ==== 步骤 2: 批量提取像素值 ====
print("开始从影像中提取特征值 (这可能需要几分钟)...")

feature_data = {} # 用于暂存提取结果

# 遍历所有 TIF 文件
for idx, tif_path in enumerate(tif_files):
    file_name = os.path.basename(tif_path)
    # 用文件名作为特征列名，方便排查
    col_name = file_name 
    
    if (idx + 1) % 10 == 0:
        print(f"  正在处理第 {idx + 1}/{len(tif_files)} 个影像...")

    with rasterio.open(tif_path) as src:
        # src.sample() 是一个生成器，我们需要将其转为 list
        # 结果形式是 [[val], [val], ...] 所以要取 [0]
        # 这里的 sample_coords 已经是转换好的坐标了
        values = [val[0] for val in src.sample(sample_coords)]
        feature_data[col_name] = values

# 将特征字典转换为 DataFrame
features_df = pd.DataFrame(feature_data)

# ==== 步骤 3: 组装最终的 train_data ====
print("正在组装训练数据...")

# 合并 基础信息(Index, Alliance) 和 提取的特征(features_df)
# 只要行索引一致，直接 concat 即可
train_data = pd.concat([points_df[['Index', 'Alliance']], features_df], axis=1)

# 处理无效值（如果点位超出了影像范围，提取值可能是 nodata）
# 假设 nodata 是 -9999 或 NaN，这里做简单清洗
# 根据你的影像实际情况修改，如果全是有效点可跳过
# train_data = train_data.replace(-9999, np.nan).dropna()

# ==== 步骤 4: 原始预处理流程衔接 (保持你原有的逻辑) ====
class_mapping = pd.read_csv(mapping_csv)

# 合并class_mapping
train_data = train_data.merge(class_mapping[['Alliance', 'num', 'Formation']], on='Alliance', how='left')

# 自动识别特征列：除了 Index, Alliance, num, Formation 之外的都是特征
feature_cols = [col for col in train_data.columns if col not in ['Index', 'Alliance', 'num', 'Formation', 'X', 'Y']]

print(f"最终训练数据构建完成: {train_data.shape}")
print(f"特征列数量: {len(feature_cols)} (应与影像数一致)")

X = train_data[feature_cols]
num_labels = train_data['num']
formation_labels = train_data['Formation']

# 编码大类标签
formation_encoder = LabelEncoder()
formation_y = formation_encoder.fit_transform(formation_labels)

X_train = X
num_train = num_labels
formation_train = formation_labels
formation_y_train = formation_y

print("数据准备完毕，可以开始训练模型。")

正在读取训练点位数据...
检测到 187 个特征影像文件。
正在进行坐标转换 (WGS84 -> 影像投影坐标)...
开始从影像中提取特征值 (这可能需要几分钟)...
  正在处理第 10/187 个影像...
  正在处理第 20/187 个影像...
  正在处理第 30/187 个影像...
  正在处理第 40/187 个影像...
  正在处理第 50/187 个影像...
  正在处理第 60/187 个影像...
  正在处理第 70/187 个影像...
  正在处理第 80/187 个影像...
  正在处理第 90/187 个影像...
  正在处理第 100/187 个影像...
  正在处理第 110/187 个影像...
  正在处理第 120/187 个影像...
  正在处理第 130/187 个影像...
  正在处理第 140/187 个影像...
  正在处理第 150/187 个影像...
  正在处理第 160/187 个影像...
  正在处理第 170/187 个影像...
  正在处理第 180/187 个影像...
正在组装训练数据...
最终训练数据构建完成: (1391, 191)
特征列数量: 187 (应与影像数一致)
数据准备完毕，可以开始训练模型。


## 构建神经网络与损失函数

In [4]:
def build_model(input_dim, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(256, activation='relu', input_dim=input_dim, kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

def multi_category_focal_loss2(gamma=2., alpha=.25, class_weights=None):
    epsilon = 1.e-7
    gamma = float(gamma)
    alpha = tf.constant(alpha, dtype=tf.float32)
    if class_weights is not None:
        weights = np.array([class_weights.get(i, 1.0) for i in range(len(class_weights))])
        class_weights_tf = tf.constant(weights, dtype=tf.float32)
    else:
        class_weights_tf = None
    def focal_loss_fixed(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        y_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        ce = -tf.math.log(y_t)
        weight = tf.pow(1. - y_t, gamma)
        fl = alpha_t * weight * ce
        if class_weights_tf is not None:
            fl = fl * class_weights_tf
        return tf.reduce_mean(fl)
    return focal_loss_fixed

def train_and_evaluate(x_train, y_train, num_classes, verbose=0):
    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
    class_weights_array = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=y_train)
    class_weights = dict(enumerate(class_weights_array))
    model = build_model(x_train.shape[1], num_classes)
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
    model.compile(loss=multi_category_focal_loss2(alpha=0.25, gamma=2, class_weights=class_weights),
                  optimizer=optimizer, metrics=['accuracy'])
    early_stopping = EarlyStopping(monitor='loss', patience=30, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=10, verbose=0)
    history = model.fit(x_train, y_train_cat, epochs=500, batch_size=64, verbose=verbose, callbacks=[reduce_lr, early_stopping])
    n_epochs = len(history.history['loss'])
    final_loss = history.history['loss'][-1]
    final_acc = history.history['acc'][-1]
    print(f"训练轮数: {n_epochs}, 最终训练精度: {final_acc:.4f}, 最终训练损失: {final_loss:.4f}")
    return model

## 训练分类模型

In [5]:
# 获取当前日期字符串
save_date = datetime.datetime.now().strftime('%Y%m%d')
model_dir = Path(f'F:/TensorFlow/xinjiang/models{save_date}')
model_dir.mkdir(parents=True, exist_ok=True)

# ==== 工具函数 ====
def get_eng_formation_map(class_mapping_path, class_mapping_df):
    if 'Eng_Formation' not in class_mapping_df.columns:
        class_mapping_df = pd.read_csv(class_mapping_path)
    return dict(zip(class_mapping_df['Formation'], class_mapping_df['Eng_Formation']))

def save_model(model, save_dir, name):
    name = name.replace(' ', '')
    path = Path(save_dir) / f'{name}.h5'
    model.save(str(path))
    print(f"模型已保存到: {path}")
    return path

def train_and_save_all_models(X_train, num_train, formation_y_train, formation_encoder, eng_formation_map, save_dir):
    # 训练大类模型
    num_formation_classes = len(np.unique(formation_y_train))
    print(f"大类（Formation）类别数: {num_formation_classes}")
    formation_model = train_and_evaluate(X_train, formation_y_train, num_formation_classes)
    print('大类训练结束')
    save_model(formation_model, save_dir, 'formation_model')

    # 训练并保存小类模型
    small_class_models = {}
    for idx, formation in enumerate(formation_encoder.classes_):
        mask = (formation_y_train == idx)
        X_sub = X_train[mask]
        y_sub = num_train[mask]
        num_encoder = LabelEncoder()
        y_sub_encoded = num_encoder.fit_transform(y_sub)
        n_classes = len(np.unique(y_sub_encoded))
        if n_classes == 1:
            print(f"大类[{formation}] 只有一个小类，无需训练模型")
            continue
        print(f"大类[{formation}]，小类数: {n_classes}")
        model = train_and_evaluate(X_sub, y_sub_encoded, n_classes)
        eng_name = eng_formation_map.get(formation, str(formation)).replace(' ', '')
        save_model(model, save_dir, f'small_class_model_{eng_name}')
        small_class_models[formation] = (model, num_encoder)
    print('全部模型保存完毕')
    return formation_model, small_class_models

# ==== 主流程 ====
# 获取保存目录
save_date = datetime.datetime.now().strftime('%Y%m%d')
model_dir = Path(f'F:/TensorFlow/xinjiang/models{save_date}')
model_dir.mkdir(parents=True, exist_ok=True)

# 获取英文名映射
gfm_path = r'F:/TensorFlow/xinjiang/traindata20250626_2_class_mapping.csv'
eng_formation_map = get_eng_formation_map(gfm_path, class_mapping)

# 训练并保存所有模型
formation_model, small_class_models = train_and_save_all_models(
    X_train, num_train, formation_y_train, formation_encoder, eng_formation_map, model_dir
)

大类（Formation）类别数: 9
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
训练轮数: 311, 最终训练精度: 0.6894, 最终训练损失: 0.0218
大类训练结束
模型已保存到: F:\TensorFlow\xinjiang\models20260202\formation_model.h5
大类[丛生草类草原]，小类数: 7
训练轮数: 465, 最终训练精度: 0.9481, 最终训练损失: 0.0205
模型已保存到: F:\TensorFlow\xinjiang\models20260202\small_class_model_TussokSteppe.h5
大类[丛生草类草甸]，小类数: 3
训练轮数: 246, 最终训练精度: 0.8571, 最终训练损失: 0.3152
模型已保存到: F:\TensorFlow\xinjiang\models20260202\small_class_model_TussokMeadow.h5
大类[农业植被] 只有一个小类，无需训练模型
大类[半乔木与灌木荒漠]，小类数: 3
训练轮数: 229, 最终训练精度: 0.8258, 最终训练损失: 0.3023
模型已保存到: F:\TensorFlow\xinjiang\models20260202\small_class_model_Semi-ArborandShurbDesert.h5
大类[半灌木与草本荒漠]，小类数: 12
训练轮数: 333, 最终训练精度: 0.8060, 最终训练损失: 0.0156
模型已保存到: F:\TensorFlow\xinjiang\models20260202\small_class_model_Semi-ShurbandHerbDesert.h5
大类[半灌木草原] 只有一个小类，无需训练模型
大类[杂类草草原] 只有一个小类，无需

### 预测

In [9]:
import rasterio
from rasterio.windows import Window
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from contextlib import ExitStack
import numpy as np
import os
import glob
import datetime

# ==== 用户配置区 ====
# 输入影像所在的文件夹路径
input_folder = r'F:\TensorFlow\CentralAsia' 

# 输出结果路径
save_date = datetime.datetime.now().strftime('%Y%m%d')
output_tif_path = f'F:/TensorFlow/xinjiang/prediction_result_{save_date}_2.tif'

# 分块大小
pred_window_size = 1024
# ====================

# 获取文件列表 (排序至关重要)
tif_files = sorted(glob.glob(os.path.join(input_folder, '*.tif')))
print(f"找到 {len(tif_files)} 个 TIF 文件。")

# 使用 ExitStack 来管理所有打开的文件句柄
with ExitStack() as stack:
    # 1. 打开第一幅影像作为【基准参考】
    # 所有的后续影像都会被强制“虚拟”成和这幅图一样的投影、坐标和尺寸
    ref_src = stack.enter_context(rasterio.open(tif_files[0]))
    
    # 获取基准的元数据
    meta = ref_src.meta.copy()
    height = ref_src.height
    width = ref_src.width
    ref_transform = ref_src.transform
    ref_crs = ref_src.crs
    
    print(f"基准影像尺寸: {width} x {height}, 坐标系: {ref_crs}")
    
    # 2. 构建虚拟影像列表 (VRT)
    # 这一步非常关键：它不会真的修改文件，而是建立一个虚拟的读取通道
    src_files_vrt = []
    
    # 将基准影像自己也放入列表
    src_files_vrt.append(ref_src) 
    
    print("正在建立虚拟对齐 (WarpedVRT)，这可能需要一点时间...")
    for fp in tif_files[1:]: # 从第2个文件开始遍历
        src = stack.enter_context(rasterio.open(fp))
        
        # 创建虚拟映射：强制让该影像拥有和基准影像一样的 CRS, Transform, Width, Height
        vrt = stack.enter_context(WarpedVRT(src,
            crs=ref_crs,
            transform=ref_transform,
            width=width,
            height=height,
            resampling=Resampling.nearest # 使用最邻近插值（分类数据）或 bilinear（连续数据）
        ))
        src_files_vrt.append(vrt)
        
    print("所有影像已虚拟对齐。开始预测...")

    # 更新元数据用于输出 (单波段，整型)
    meta.update(
        dtype=rasterio.int32,
        count=1,
        nodata=-9999,
        compress='lzw'
    )

    # 3. 创建输出文件并开始分块处理
    with rasterio.open(output_tif_path, 'w', **meta) as dst:
        # 基于【基准影像】生成窗口
        windows = [window for ij, window in ref_src.block_windows(1)]
        total_windows = len(windows)
        
        for i, window in enumerate(windows):
            if (i + 1) % 10 == 0:
                print(f"正在处理分块: {i + 1} / {total_windows}")

            try:
                # 4. 读取数据
                # 因为用了 VRT，所有 src.read 都会返回严格一致的尺寸
                # 即使原始图只有一半大，VRT 也会自动用 nodata 补齐
                chunk_stack = np.stack([vrt.read(1, window=window) for vrt in src_files_vrt])
                
            except Exception as e:
                print(f"分块 {i+1} 读取依然失败: {e}")
                continue

            # 5. 数据重塑与预测 (逻辑与之前相同)
            bands, w_h, w_w = chunk_stack.shape
            reshaped_data = chunk_stack.transpose(1, 2, 0).reshape(-1, bands)
            
            # 初始化结果
            final_pred_chunk = np.full(reshaped_data.shape[0], -9999, dtype=np.int32)
            
            # 简单掩膜：只要有一个波段是NaN，就视为无效
            # 如果你的数据里 nodata 是数值(如-9999)，请改为 (reshaped_data != -9999).all(axis=1)
            valid_mask = ~np.isnan(reshaped_data).any(axis=1)
            
            if np.sum(valid_mask) > 0:
                X_chunk = reshaped_data[valid_mask]
                
                # ---- 预测逻辑 (复制之前的逻辑) ----
                formation_prob = formation_model.predict(X_chunk, verbose=0)
                formation_pred = np.argmax(formation_prob, axis=1)
                
                chunk_alliance_pred = np.zeros_like(formation_pred, dtype=np.int32)
                unique_formations = np.unique(formation_pred)
                
                for fmt_idx in unique_formations:
                    formation_name = formation_encoder.classes_[fmt_idx]
                    current_fmt_mask = (formation_pred == fmt_idx)
                    
                    if formation_name in small_class_models:
                        sub_model, num_encoder = small_class_models[formation_name]
                        X_sub = X_chunk[current_fmt_mask]
                        sub_prob = sub_model.predict(X_sub, verbose=0)
                        sub_pred_idx = np.argmax(sub_prob, axis=1)
                        sub_pred_val = num_encoder.inverse_transform(sub_pred_idx)
                        chunk_alliance_pred[current_fmt_mask] = sub_pred_val
                        
                    elif fmt_idx in single_class_map:
                        default_val = single_class_map[fmt_idx]
                        chunk_alliance_pred[current_fmt_mask] = default_val
                    else:
                        chunk_alliance_pred[current_fmt_mask] = -9999

                final_pred_chunk[valid_mask] = chunk_alliance_pred
                # --------------------------------

            # 写入结果
            result_2d = final_pred_chunk.reshape(w_h, w_w)
            dst.write(result_2d, 1, window=window)

print(f"全部完成！结果已保存至: {output_tif_path}")

找到 187 个 TIF 文件。
基准影像尺寸: 18177 x 9044, 坐标系: EPSG:4326
正在建立虚拟对齐 (WarpedVRT)，这可能需要一点时间...
所有影像已虚拟对齐。开始预测...
正在处理分块: 10 / 2592
正在处理分块: 20 / 2592
正在处理分块: 30 / 2592
正在处理分块: 40 / 2592
正在处理分块: 50 / 2592
正在处理分块: 60 / 2592
正在处理分块: 70 / 2592
正在处理分块: 80 / 2592
正在处理分块: 90 / 2592
正在处理分块: 100 / 2592
正在处理分块: 110 / 2592
正在处理分块: 120 / 2592
正在处理分块: 130 / 2592
正在处理分块: 140 / 2592
正在处理分块: 150 / 2592
正在处理分块: 160 / 2592
正在处理分块: 170 / 2592
正在处理分块: 180 / 2592
正在处理分块: 190 / 2592
正在处理分块: 200 / 2592
正在处理分块: 210 / 2592
正在处理分块: 220 / 2592
正在处理分块: 230 / 2592
正在处理分块: 240 / 2592
正在处理分块: 250 / 2592
正在处理分块: 260 / 2592
正在处理分块: 270 / 2592
正在处理分块: 280 / 2592
正在处理分块: 290 / 2592
正在处理分块: 300 / 2592
正在处理分块: 310 / 2592
正在处理分块: 320 / 2592
正在处理分块: 330 / 2592
正在处理分块: 340 / 2592
正在处理分块: 350 / 2592
正在处理分块: 360 / 2592
正在处理分块: 370 / 2592
正在处理分块: 380 / 2592
正在处理分块: 390 / 2592
正在处理分块: 400 / 2592
正在处理分块: 410 / 2592
正在处理分块: 420 / 2592
正在处理分块: 430 / 2592
正在处理分块: 440 / 2592
正在处理分块: 450 / 2592
正在处理分块: 460 / 2592
正在处理分块: 470 / 2592
正在处理分块: 48